In [ ]:
LINEAS = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]
ORDEN = [4,0,2,6,8,1,3,5,7]


def nuevo_juego():
    celdas = []
    for i in range(9):
        celdas.append([' '] * 9)
    juego = {'celdas': celdas, 'locales': [' '] * 9, 'siguiente': None, 'turno': 'X', 'resultado': None, 'historial': []}
    return juego


def rival_de(ficha):
    if ficha == 'X':
        return 'O'
    return 'X'


def quien_gana(lista):
    for linea in LINEAS:
        a = lista[linea[0]]
        b = lista[linea[1]]
        c = lista[linea[2]]
        if a != ' ' and a != 'D' and a == b and b == c:
            return a
    return ''


def jugadas_legales(juego):
    jugadas = []
    if juego['resultado'] != None:
        return jugadas
    tableros = []
    if juego['siguiente'] == None:
        for b in ORDEN:
            if juego['locales'][b] == ' ':
                tableros.append(b)
    else:
        tableros.append(juego['siguiente'])
    for b in tableros:
        for c in ORDEN:
            if juego['celdas'][b][c] == ' ':
                jugadas.append([b, c])
    return jugadas


def jugar(juego, b, c):
    juego['historial'].append([b, c, juego['siguiente'], juego['locales'][b], juego['resultado']])
    juego['celdas'][b][c] = juego['turno']
    ganador = quien_gana(juego['celdas'][b])
    if ganador != '':
        juego['locales'][b] = ganador
    elif ' ' not in juego['celdas'][b]:
        juego['locales'][b] = 'D'
    if juego['locales'][c] == ' ':
        juego['siguiente'] = c
    else:
        juego['siguiente'] = None
    ganador = quien_gana(juego['locales'])
    if ganador != '':
        juego['resultado'] = ganador
    elif ' ' not in juego['locales']:
        juego['resultado'] = 'D'
    juego['turno'] = rival_de(juego['turno'])


def deshacer(juego):
    ultimo = juego['historial'].pop()
    juego['celdas'][ultimo[0]][ultimo[1]] = ' '
    juego['siguiente'] = ultimo[2]
    juego['locales'][ultimo[0]] = ultimo[3]
    juego['resultado'] = ultimo[4]
    juego['turno'] = rival_de(juego['turno'])


def contar_lineas(lista, yo):
    rival = rival_de(yo)
    total = 0
    for linea in LINEAS:
        trio = [lista[linea[0]], lista[linea[1]], lista[linea[2]]]
        if trio.count(' ') == 1:
            if trio.count(yo) == 2:
                total = total + 1
            if trio.count(rival) == 2:
                total = total - 1
    return total


def evaluar(juego, yo):
    rival = rival_de(yo)
    puntos = 0
    for b in range(9):
        if juego['locales'][b] == yo:
            puntos = puntos + 10
        if juego['locales'][b] == rival:
            puntos = puntos - 10
        if juego['locales'][b] == ' ':
            puntos = puntos + contar_lineas(juego['celdas'][b], yo)
    puntos = puntos + 30 * contar_lineas(juego['locales'], yo)
    return puntos


def minimax(juego, profundidad, yo, alfa, beta):
    if juego['resultado'] == yo:
        return 1000 + profundidad
    if juego['resultado'] == rival_de(yo):
        return -1000 - profundidad
    if juego['resultado'] == 'D':
        return 0
    if profundidad == 0:
        return evaluar(juego, yo)
    if juego['turno'] == yo:
        mejor = -100000
        for jugada in jugadas_legales(juego):
            jugar(juego, jugada[0], jugada[1])
            valor = minimax(juego, profundidad - 1, yo, alfa, beta)
            deshacer(juego)
            if valor > mejor:
                mejor = valor
            if mejor > alfa:
                alfa = mejor
            if alfa >= beta:
                break
    else:
        mejor = 100000
        for jugada in jugadas_legales(juego):
            jugar(juego, jugada[0], jugada[1])
            valor = minimax(juego, profundidad - 1, yo, alfa, beta)
            deshacer(juego)
            if valor < mejor:
                mejor = valor
            if mejor < beta:
                beta = mejor
            if alfa >= beta:
                break
    return mejor


def mejor_jugada(juego, profundidad):
    yo = juego['turno']
    mejor_valor = -100000
    mejor = None
    for jugada in jugadas_legales(juego):
        jugar(juego, jugada[0], jugada[1])
        valor = minimax(juego, profundidad - 1, yo, -100000, 100000)
        deshacer(juego)
        if valor > mejor_valor:
            mejor_valor = valor
            mejor = jugada
    return mejor


import ipywidgets as widgets
from IPython.display import display

partida = {'juego': None, 'humano': 'X', 'nivel': 3, 'ultima': None}
botones = []
mensaje = widgets.Label(value='')


def dibujar():
    juego = partida['juego']
    mi_turno = juego['resultado'] == None and juego['turno'] == partida['humano']
    activos = []
    if mi_turno:
        for jugada in jugadas_legales(juego):
            activos.append(jugada[0])
    for b in range(9):
        for c in range(9):
            boton = botones[b][c]
            ficha = juego['celdas'][b][c]
            boton.description = ficha.strip()
            if juego['locales'][b] == 'X':
                color = '#c7d4ff'
            elif juego['locales'][b] == 'O':
                color = '#ffc7c7'
            elif juego['locales'][b] == 'D':
                color = '#d9d9d9'
            elif b in activos:
                color = '#fff3b0'
            else:
                color = '#f2f2f2'
            if ficha == 'X':
                color = '#6f8cff'
            if ficha == 'O':
                color = '#ff7f7f'
            if partida['ultima'] == [b, c]:
                color = '#ffb347'
            boton.style.button_color = color
    if juego['resultado'] == 'D':
        mensaje.value = 'Empate'
    elif juego['resultado'] == partida['humano']:
        mensaje.value = '¡Ganaste!'
    elif juego['resultado'] != None:
        mensaje.value = 'Gana la máquina'
    elif mi_turno:
        if juego['siguiente'] == None:
            mensaje.value = 'Tu turno: juega en cualquier tablero'
        else:
            mensaje.value = 'Tu turno: juega en el tablero ' + str(juego['siguiente'] + 1)
    else:
        mensaje.value = 'La máquina está pensando...'


def turno_maquina():
    juego = partida['juego']
    if juego['resultado'] == None and juego['turno'] != partida['humano']:
        dibujar()
        jugada = mejor_jugada(juego, partida['nivel'])
        jugar(juego, jugada[0], jugada[1])
        partida['ultima'] = jugada
    dibujar()


def al_hacer_clic(boton):
    juego = partida['juego']
    jugada = [boton.tab, boton.cas]
    if juego['resultado'] != None or juego['turno'] != partida['humano']:
        return
    if jugada not in jugadas_legales(juego):
        return
    jugar(juego, jugada[0], jugada[1])
    partida['ultima'] = jugada
    turno_maquina()


def nueva_partida(boton):
    partida['juego'] = nuevo_juego()
    partida['humano'] = lista_ficha.value
    partida['nivel'] = lista_nivel.value
    partida['ultima'] = None
    turno_maquina()


lista_ficha = widgets.Dropdown(options=['X', 'O'], value='X', description='Juegas con:')
lista_nivel = widgets.Dropdown(options=[('Fácil', 2), ('Normal', 3), ('Difícil', 4)], value=3, description='Nivel:')
boton_nuevo = widgets.Button(description='Nueva partida', button_style='success')
boton_nuevo.on_click(nueva_partida)

cajas = []
for b in range(9):
    fila = []
    for c in range(9):
        boton = widgets.Button(description='', layout=widgets.Layout(width='42px', height='42px', margin='1px', padding='0px'))
        boton.tab = b
        boton.cas = c
        boton.on_click(al_hacer_clic)
        fila.append(boton)
    botones.append(fila)
    cajas.append(widgets.GridBox(fila, layout=widgets.Layout(grid_template_columns='repeat(3, 44px)', border='3px solid black', margin='3px')))

tablero = widgets.GridBox(cajas, layout=widgets.Layout(grid_template_columns='repeat(3, auto)'))

display(widgets.HBox([lista_ficha, lista_nivel, boton_nuevo]), mensaje, tablero)
nueva_partida(None)


Label(value='')

GridBox(children=(GridBox(children=(Button(layout=Layout(height='42px', margin='1px', padding='0px', width='42…